In [3]:
# Sydney Housing Project - V2.0 Feature Engineering (Spatial)
import pandas as pd
import numpy as np
from clearml import Task

# 1. Initialize ClearML tracker for V2
task = Task.init(project_name="Sydney_Housing_Project", task_name="V2_Feature_Engineering_Distance")

print(" Loading cleaned V1 rental data...")
df = pd.read_csv("nsw_rental_bonds.csv")

# 2. Fetch standard Australian Postcode mapping data directly from an open-source GitHub repo
print(" Fetching Australian Postcode spatial mapping...")
url = "https://raw.githubusercontent.com/matthewproctor/australianpostcodes/master/australian_postcodes.csv"
postcode_df = pd.read_csv(url)

# 3. Filter for NSW postcodes and fix the column name ('long' -> 'lon')
nsw_postcodes = postcode_df[postcode_df['state'] == 'NSW'][['postcode', 'lat', 'long']].drop_duplicates(subset=['postcode'])
nsw_postcodes = nsw_postcodes.rename(columns={'long': 'lon'})

# 4. Merge our rental data with the coordinates based on Postcode
print(" Merging rental data with spatial coordinates...")
df = df.merge(nsw_postcodes, left_on='Postcode', right_on='postcode', how='left')

# 5. The Haversine Formula (Calculates great-circle distance between two points on a sphere)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0 # Earth radius in kilometers
    dLat = np.radians(lat2 - lat1)
    dLon = np.radians(lon2 - lon1)
    a = np.sin(dLat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dLon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

# 6. Calculate distance to Sydney Central Station
print(" Calculating distance to Sydney Central Station (km)...")

# location of sydney central station
central_station_lat, central_station_lon = -33.8832, 151.2069 

# new list Distance_to_Central_km
df['Distance_to_Central_km'] = haversine(df['lat'], df['lon'], central_station_lat, central_station_lon)

# 7. Clean up redundant columns we don't need anymore
df = df.drop(columns=['postcode', 'lat', 'lon'])

print("\n V2.0 Features generated successfully!")
display(df.head())

 Loading cleaned V1 rental data...
 Fetching Australian Postcode spatial mapping...
 Merging rental data with spatial coordinates...
 Calculating distance to Sydney Central Station (km)...

 V2.0 Features generated successfully!


,Lodgement Date,Postcode,Dwelling Type,Bedrooms,Weekly Rent,Distance_to_Central_km
0,23/03/2026,2000,F,0,850,2.569335
1,03/03/2026,2000,F,0,850,2.569335
2,19/03/2026,2000,F,0,560,2.569335
3,20/03/2026,2000,F,0,850,2.569335
4,12/03/2026,2000,F,0,550,2.569335
